# Phase 9 — Hyperparameter Optimization Framework

An interactive educational walkthrough of **Phase 9: Hyperparameter Optimization (`v0.9.0`)** for Motor Imagery EEG Classification.
This notebook demonstrates search space definitions, global parameter & strategy registries (`PARAMETER_REGISTRY`, `STRATEGY_REGISTRY`), early search space validation (`SearchSpaceValidator`), trial scheduling (`TrialScheduler`), optimization execution (`HyperparameterOptimizer`), visualization plots (Optimization History & Parallel Coordinates), and export of reproducible artifacts (`best_config.yaml`, `leaderboard.csv`, `summary.json`, `manifest.json`).

## 1. Objective & Design Philosophy

Phase 9 establishes an extensible, plugin-based Hyperparameter Optimization (HPO) engine following the **Open/Closed Principle** (*"Open for extension, closed for modification"*).

- **Zero Core Modifications**: Connects around `Trainer`, `EEGMotorImageryModel`, and `Evaluator` without altering any existing code.
- **Dependency Flow**:
  `Config (configs/hpo.yaml) -> Factory -> Registries -> Strategy Interface -> Implementation`
- **Traceability**: Every trial generates an immutable trial record, trial manifest, and output checkpoint.

## 2. Framework Package Structure

```text
configs/
└── hpo.yaml                     [Search space & optimization budget config]

hpo/
├── registry.py                  [PARAMETER_REGISTRY & STRATEGY_REGISTRY]
├── parameters/                  [FloatParameter, IntegerParameter, CategoricalParameter, LogUniformParameter]
├── search_space.py              [SearchSpace parser]
├── validator.py                 [SearchSpaceValidator]
├── trial.py                     [TrialStatus Enum & immutable Trial dataclass]
├── objective.py                 [ObjectiveResult dataclass]
├── scheduler.py                 [TrialScheduler execution queue]
├── strategies/                  [RandomSearchStrategy, GridSearchStrategy, OptunaSearchStrategy]
├── factory.py                   [build_hpo_strategy factory]
├── optimizer.py                 [HyperparameterOptimizer engine]
├── results.py                   [HPOResultsManager]
└── visualization/               [plot_optimization_history & plot_parallel_coordinates]

scripts/
└── optimize.py                  [CLI Entry Point]
```

## 3. Implementation Imports & Registries Inspection

In [ ]:
import os
import sys
import torch
import pandas as pd
import matplotlib.pyplot as plt

def get_project_root():
    curr = os.path.abspath(os.getcwd())
    while curr and not os.path.exists(os.path.join(curr, "models")):
        parent = os.path.dirname(curr)
        if parent == curr:
            break
        curr = parent
    return curr

PROJECT_ROOT = get_project_root()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"[OK] Project Root set to: {PROJECT_ROOT}")

from configs.config_loader import load_master_config
from hpo import (
    PARAMETER_REGISTRY,
    STRATEGY_REGISTRY,
    SearchSpace,
    SearchSpaceValidator,
    TrialScheduler,
    HyperparameterOptimizer,
    build_hpo_strategy,
)
from hpo.visualization import plot_optimization_history, plot_parallel_coordinates

print("[OK] Registered Parameter Types:", list(PARAMETER_REGISTRY.keys()))
print("[OK] Registered Search Strategies:", list(STRATEGY_REGISTRY.keys()))

## 4. Search Space Validation & Sampling Demo

In [ ]:
search_space_cfg = {
    "learning_rate": {"type": "loguniform", "low": 1e-5, "high": 1e-3},
    "weight_decay": {"type": "float", "distribution": "uniform", "low": 0.0001, "high": 0.05},
    "dropout": {"type": "float", "distribution": "uniform", "low": 0.1, "high": 0.5},
    "batch_size": {"type": "categorical", "values": [16, 32, 64]},
    "d_model": {"type": "categorical", "values": [32, 64, 128]},
}

# Validate search space configuration
SearchSpaceValidator.validate_search_space_config(search_space_cfg)
search_space = SearchSpace(search_space_cfg)

print("Sampled Hyperparameter Combination:")
sampled_params = search_space.sample()
for k, v in sampled_params.items():
    print(f"  - {k:<15}: {v}")

## 5. Hyperparameter Optimization Execution (3-Trial Demo)

Executing `HyperparameterOptimizer` for 3 trials with `RandomSearchStrategy`.

In [ ]:
master_cfg = load_master_config(project_root=PROJECT_ROOT)
out_dir = os.path.join(PROJECT_ROOT, "outputs", "hpo")

master_cfg["hpo"] = {
    "seed": 42,
    "strategy": "random",
    "metric": "val_accuracy",
    "mode": "max",
    "n_trials": 3,
    "max_epochs_per_trial": 1,
    "output_dir": out_dir,
}
master_cfg["search_space"] = search_space_cfg

optimizer = HyperparameterOptimizer(master_cfg)
scheduler = optimizer.optimize(resume=False)

best_trial = scheduler.get_best_trial()
print(f"\n=== Optimization Finished ===")
print(f"Best Trial ID:    #{best_trial.trial_id}")
print(f"Best Val Score:   {best_trial.score:.4f}")
print(f"Best Parameters:  {best_trial.params}")

## 6. Optimization Visualizations & Artifact Verification

### 6.1 Optimization History Plot

In [ ]:
trials_csv = os.path.join(out_dir, "trials.csv")
fig_hist = plot_optimization_history(trials_csv, metric_name="val_accuracy")
plt.show()

### 6.2 Parallel Coordinates Hyperparameter Interaction Plot

In [ ]:
fig_par = plot_parallel_coordinates(trials_csv)
plt.show()

### 6.3 Exported Artifacts Inspection

In [ ]:
df_leaderboard = pd.read_csv(os.path.join(out_dir, "leaderboard.csv"))
print("\n--- HPO Leaderboard ---")
display(df_leaderboard)

## 7. Conclusion & Research Framework Roadmap

### Key Takeaways:
1. **Open/Closed Plugin Architecture**: `hpo/` package adds full optimization capabilities via registries and strategy interfaces without altering core model or trainer code.
2. **Search Space Validation**: `SearchSpaceValidator` catches invalid bounds, negative loguniform ranges, and empty categoricals prior to execution.
3. **State & Queue Management**: `TrialScheduler` tracks trial status (`PENDING -> RUNNING -> COMPLETED`) and enables clean resumption.
4. **Complete Output Artifacts**: Exports `leaderboard.csv`, `best_config.yaml`, `summary.json`, `trials.csv`, `optimization_history.png`, `parallel_coordinates.png`, and per-trial `manifest.json` files.

**Phase 9 is complete and fully validated.** The framework is ready for **Phase 10 (Data Augmentation - WGAN-GP)**.